# 0.1. Delete baseline

This baseline is based on the idea of simply deleting profanity from the text. This is a very simple baseline, but it is a good starting point for the task.


In [2]:
import pandas as pd

In [3]:
# Load the "bad bad words" dataset
bad_bad_words = pd.read_csv("../data/external/bad-words.csv", names=["word"])
bad_bad_words.head()

,word
0,jigaboo
1,mound of venus
2,asslover
3,s&m
4,queaf


We will use the kgram similarity to find profanity in the text. If the similarity between a word and some word in the profanity dataset is greater than a threshold, we will delete the word.


In [4]:
def kgram(word: str, k: int = 3) -> list:
    """Return a list of k-grams for a given word."""
    return [word[i : i + k] for i in range(len(word) - k + 1)]


def kgram_similarity(word1: str, word2: str, k: int = 3) -> float:
    """Return the Jaccard similarity between two words."""
    A = set(kgram(word1, k))
    B = set(kgram(word2, k))

    inter_len = len(A & B)
    union_len = len(A | B)

    if union_len == 0:
        return 0

    return inter_len / union_len


def is_bad_word(word: str) -> bool:
    """Use k-gram similarity to determine if a word is bad."""
    return any(
        kgram_similarity(word, bad_word, 3) > 0.5 for bad_word in bad_bad_words["word"]
    )


def clean_word(word: str) -> str:
    """Remove symbols from a word."""
    cleaned_word = "".join(c for c in word if c.isalpha())
    cleaned_word = cleaned_word.strip()
    return cleaned_word


def detoxify(text: str, replace_with: str = "*") -> str:
    """Replace bad words in a text with a replacement string."""
    words = text.split()

    for i, word in enumerate(words):
        cleaned_word = clean_word(word).lower()

        if is_bad_word(cleaned_word):
            words[i] = replace_with * len(word)

    return " ".join(words)

In [5]:
# Get random toxic sentence from the dataset
df_test = pd.read_csv(
    "../data/interim/processed.tsv", sep="\t", header=None, names=["tox", "detox"]
)
df_test.head()

,tox,detox
0,"if Alkar floods her with her mental waste, it ...","If Alkar is flooding her with psychic waste, t..."
1,you're becoming disgusting.,Now you're getting nasty.
2,"well, we can spare your life.","Well, we could spare your life, for one."
3,"monkey, you have to wake up.","Ah! Monkey, you've got to snap out of it."
4,I have orders to kill her.,I've got orders to put her down.


In [6]:
import random

In [15]:
tox_infer = df_test["tox"][random.randint(0, len(df_test))]

print(f"Tox:\t{tox_infer}\n\nDelete:\t{detoxify(tox_infer)}")

Tox:	What the hell do they expect me to say?

Delete:	What the **** do they expect me to say?


## Test the baseline


In [8]:
import warnings

warnings.filterwarnings("ignore")

from tqdm import tqdm

In [9]:
# Dynamicly load evaluation metrics
%run -i ../src/models/evaluation.py

In [10]:
# Instantiate metric classes
similarity = Similarity()
toxicity = STAToxic()

In [11]:
# Load test dataset
df_test = pd.read_csv(
    "../data/interim/test.tsv",
    sep="\t",
    header=None,
    names=["tox", "detox"],
)
df_test.head()

,tox,detox
0,I stood up with a guy who called me a fag.,And stood up to a boy who called me queer.
1,"Hello, morons.","hello, maths."
2,"shut up, dear sister!","Be quiet, dear sister!"
3,"I'll do what you do to me, your own gun.",I use your own tech to do to you what you did ...
4,"Uh, Derek's an ass who won't return my calls.",Tizzler Derek doesn't answer my calls.


In [12]:
df_delete_scores = pd.DataFrame(columns=["wo", "cs", "bleu", "detox_score"])

preds = []

for i in tqdm(range(len(df_test))):
    prompt = df_test["tox"][i]

    pred = detoxify(prompt)
    preds.append(pred)

    # Calculate metrics
    wo = similarity.get_wo_score(prompt, pred)
    cs = similarity.get_cosine_score(prompt, pred)
    bleu = similarity.get_bleu_score(prompt, pred)

    # Append to dataframe
    df_delete_scores.loc[i] = [wo, cs, bleu, 0]

toxicity_report = toxicity.toxicity_report(preds)
df_delete_scores["detox_score"] = 1 - toxicity_report[["toxic"]].mean(axis=1)

df_delete_scores.head()

100%|██████████| 100/100 [00:08<00:00, 11.79it/s]


,wo,cs,bleu,detox_score
0,0.818182,0.782331,9.011551e-01,0.991979
1,0.000000,-0.074898,9.418382e-232,0.962162
2,0.333333,0.163804,6.395028e-01,0.066541
3,0.461538,0.854835,8.960111e-01,0.962162
4,0.250000,0.826650,8.953712e-01,0.988067


In [13]:
# Save scores
df_delete_scores.to_csv("../data/interim/delete_baseline_scores.csv", index=False)